# Is it a Bird?

The opening exercise from fast.ai's [Practical Deep Learning for Coders](https://course.fast.ai/):
build an image classifier that can tell a bird from a forest, using images scraped from
the web rather than a prepared dataset.

The point of the exercise is how little is needed. There is no hand-labelling step and no
model architecture to design — the folder a picture is downloaded into *is* its label, and
the model is a pretrained `resnet18` fine-tuned for three epochs.

The helper below wraps DuckDuckGo image search with a retry-and-backoff loop, because the
search endpoint rate-limits aggressively when you pull several hundred images in a row.

In [ ]:
!pip install -Uq ddgs
from ddgs import DDGS
from fastcore.all import *
from fastai.vision.all import *
from fastdownload import download_url
import time, random

def search_images(term, max_images=30, retries=5):
    print(f"Searching for '{term}'")
    for attempt in range(retries):
        try:
            with DDGS() as ddgs:
                results = ddgs.images(query=term, max_results=max_images)
                return L([r['image'] for r in results])
        except Exception as e:
            wait = 15 * (attempt + 1) + random.randint(5, 15)
            print(f"  {e.__class__.__name__}, waiting {wait}s (attempt {attempt+1}/{retries})")
            time.sleep(wait)
    print(f"  Failed after {retries} retries")
    return L([])


## Check the search works

Before downloading hundreds of images, pull a single result and look at it. If the search
helper is broken or rate-limited, it is much easier to see that here than halfway through
building the dataset.

In [ ]:
urls = search_images('bird photos', max_images=10)
urls[0]


In [ ]:
download_url(urls[0], 'bird.jpg', show_progress=False)
im = Image.open('bird.jpg')
im.to_thumb(256,256)


In [ ]:
forest_urls = search_images('forest photos', max_images=10)
download_url(forest_urls[0], 'forest.jpg', show_progress=False)
Image.open('forest.jpg').to_thumb(256,256)

## Build the dataset

Download roughly 50 images per category into `bird_or_not/forest` and `bird_or_not/bird`.
The directory name is what labels the data later, so the folder layout matters.

Images are resized to a maximum of 400 px on download. Training will resize them again
anyway, and keeping the originals wastes disk and slows every epoch.

In [ ]:
searches = 'forest','bird'
path = Path('bird_or_not')
for o in searches:
    dest = (path/o)
    dest.mkdir(exist_ok=True, parents=True)
    download_images(dest, urls=search_images(f'{o} photo', max_images=50))
    time.sleep(15 + random.randint(5, 15))
    resize_images(path/o, max_size=400, dest=path/o)

## Discard broken downloads

Some fraction of any web scrape will be dead links, HTML error pages saved as `.jpg`, or
truncated files. `verify_images` finds them and they get unlinked — one corrupt file is
enough to crash training partway through.

In [ ]:
failed = verify_images(get_image_files(path))
failed.map(Path.unlink)
len(failed)

## Assemble the DataLoaders

The `DataBlock` describes the pipeline rather than performing it: inputs are images and
targets are categories, the items come from the file listing, the label is the parent
folder name, and 20% is held back for validation with a fixed seed so the split is
reproducible.

Every image is squished to 192x192 so they can be batched together.

In [ ]:
dls = DataBlock(
    blocks=(ImageBlock, CategoryBlock), 
    get_items=get_image_files, 
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=parent_label,
    item_tfms=[Resize(192, method='squish')]
).dataloaders(path, bs=32)

dls.show_batch(max_n=6)

## Train

`vision_learner` starts from a `resnet18` already trained on ImageNet, so it arrives
knowing about edges, textures and shapes. Fine-tuning only has to teach it the distinction
between these two categories, which is why three epochs on a few hundred images is enough.

In [ ]:
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(3)

## Predict

Finally, run the trained model against the single forest image downloaded earlier, and
print the predicted category along with the probability.

In [ ]:
is_bird,_,probs = learn.predict(PILImage.create('forest.jpg'))
print(f"This is a: {is_bird}.")
print(f"Probability it's a bird: {probs[0]:.4f}")